# Patch 16.19 OTP Analysis — Exploratory Data Analysis

This notebook uses one row per sampled Master+ player-match observation. `otp_rate` measures the share of the player's 50 completed Ranked Solo/Duo games **before** that match played on the selected champion. It filters to LoLalytics matchups with at least 50 games, uses the raw Global D2+ matchup win rate, and cleans the dataframe only in memory.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
sns.set_theme(style='whitegrid', context='notebook')

def display_percentages(frame, columns, digits=2, signed=()):
    """Display percentage columns cleanly without requiring pandas Styler/Jinja2."""
    output = frame.copy()
    for column in columns:
        if column in output:
            sign = '+' if column in signed else ''
            output[column] = output[column].map(
                lambda value: f'{value:{sign}.{digits}%}' if pd.notna(value) else ''
            )
    display(output)

## 1. Load and clean the analysis dataframe

The complete merged dataset remains unchanged on disk. Before applying the 50-game matchup cutoff, this section checks how many observations the cutoff retains by rank, lane, and previous-50 OTP rate. `df` is filtered and reorganized only in memory.


In [ ]:
start_dir = Path.cwd().resolve()
search_dirs = [start_dir, *start_dir.parents]
PROJECT_DIR = next(
    (path for path in search_dirs if (path / 'data' / 'processed').exists()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find the repository data/processed directory.')

DATA_PATH = PROJECT_DIR / 'data' / 'processed' / 'otp_analysis_16_19.parquet'
raw_df = pd.read_parquet(DATA_PATH)

required = {
    'match_id', 'puuid', 'opponent_puuid', 'game_start_timestamp',
    'seed_tier', 'side', 'lane', 'champion', 'opponent_champion', 'win',
    'champion_games', 'player_games', 'champion_share', 'otp_measure',
    'raw_matchup_winrate', 'matchup_games', 'matchup_games_ge_50'
}
missing = required.difference(raw_df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')
if not raw_df['otp_measure'].eq('previous_50_ranked_solo').all() or not raw_df['player_games'].eq(50).all():
    raise ValueError('Rerun the collection and merge notebooks to build pre-game previous-50 OTP rates.')

# Diagnose which observations are removed by the LoLalytics sample-size rule.
retained_mask = raw_df['matchup_games_ge_50'].fillna(False).astype(bool)
cutoff_audit = raw_df[['seed_tier', 'lane', 'champion_share']].copy()
cutoff_audit['retained'] = retained_mask
cutoff_audit['otp_group'] = pd.cut(
    pd.to_numeric(cutoff_audit['champion_share'], errors='coerce'),
    bins=[0, .20, .40, .60, .80, 1.000001],
    labels=['0–20%', '20–40%', '40–60%', '60–80%', '80–100%'],
    include_lowest=True, right=False,
)

def show_cutoff_counts(group_column):
    counts = pd.crosstab(
        cutoff_audit[group_column].astype('string').fillna('missing'),
        cutoff_audit['retained'],
    ).reindex(columns=[True, False], fill_value=0)
    counts.columns = ['retained_rows', 'excluded_rows']
    counts['retained_pct'] = counts['retained_rows'] / counts.sum(axis=1)
    display_percentages(counts, ['retained_pct'], digits=1)

print(f'50-game matchup cutoff: {retained_mask.sum():,} of {len(raw_df):,} '
      f'player rows retained ({retained_mask.mean():.1%}); '
      f'{(~retained_mask).sum():,} excluded.')
print('Cutoff by sampled player rank:')
show_cutoff_counts('seed_tier')
print('Cutoff by lane:')
show_cutoff_counts('lane')
print('Cutoff by previous-50 OTP-rate group:')
show_cutoff_counts('otp_group')

df = raw_df.loc[retained_mask].copy()
df = df.rename(columns={
    'seed_tier': 'rank',
    'champion_share': 'otp_rate',
})
df['matchup_winrate'] = pd.to_numeric(df['raw_matchup_winrate'], errors='coerce') / 100
df['game_start_time'] = (
    pd.to_datetime(df['game_start_timestamp'], unit='ms', utc=True)
    .dt.tz_convert('America/Los_Angeles')
)
df['win'] = df['win'].astype(bool)

rank_order = ['master', 'grandmaster', 'challenger']
lane_order = ['top', 'jungle', 'mid', 'adc', 'support']
df['rank'] = pd.Categorical(df['rank'], categories=rank_order, ordered=True)
df['lane'] = pd.Categorical(df['lane'], categories=lane_order, ordered=True)

analysis_columns = [
    'match_id', 'puuid', 'opponent_puuid', 'game_start_time',
    'rank', 'side', 'lane', 'champion', 'opponent_champion', 'win',
    'champion_games', 'player_games', 'otp_rate',
    'matchup_winrate', 'matchup_games',
]
df = (
    df.loc[:, analysis_columns]
    .sort_values(['game_start_time', 'match_id', 'puuid'])
    .reset_index(drop=True)
)

print('Source:', DATA_PATH)
print('Cleaned shape:', df.shape)
display(df.head())


## 2. Data-quality audit

In [ ]:
audit = pd.Series({
    'player rows': len(df),
    'unique matches': df['match_id'].nunique(),
    'unique sampled players': df['puuid'].nunique(),
    'duplicate player-match rows': df.duplicated(['match_id', 'puuid']).sum(),
    'overall actual win rate': df['win'].mean(),
    'average raw matchup win rate': df['matchup_winrate'].mean(),
    'earliest match': df['game_start_time'].min(),
    'latest match': df['game_start_time'].max(),
})
display(audit.to_frame('value'))

missingness = (
    df.isna().sum().rename('missing_rows').to_frame()
    .assign(missing_pct=lambda table: table['missing_rows'] / len(df))
    .sort_values('missing_rows', ascending=False)
)
display(missingness)

assert df.duplicated(['match_id', 'puuid']).sum() == 0
assert df['matchup_games'].ge(50).all()

## 3. Rank distribution

The player count gives each sampled player one vote; the observation count shows how much each rank contributes to the player-match dataset.

In [ ]:
unique_player_ranks = df[['puuid', 'rank']].drop_duplicates()
rank_distribution = (
    unique_player_ranks.groupby('rank', observed=True)['puuid'].nunique()
    .rename('unique_players').to_frame()
    .join(df.groupby('rank', observed=True).size().rename('player_rows'))
    .assign(
        player_pct=lambda table: table['unique_players'] / table['unique_players'].sum(),
        row_pct=lambda table: table['player_rows'] / table['player_rows'].sum(),
    )
)
display_percentages(rank_distribution, ['player_pct', 'row_pct'], digits=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=rank_distribution.reset_index(), x='rank', y='unique_players', ax=axes[0])
axes[0].set(title='Unique sampled players by rank', xlabel='Rank', ylabel='Players')
sns.barplot(data=rank_distribution.reset_index(), x='rank', y='player_rows', ax=axes[1])
axes[1].set(title='Player-match observations by rank', xlabel='Rank', ylabel='Rows')
plt.tight_layout()
plt.show()

## 4. Actual win rate versus LoLalytics matchup win rate

Matchups are divided into ten similarly sized groups. Points on the dashed line indicate where observed player win rate equals the raw Global D2+ matchup rate. This is a descriptive comparison: players and matches can appear more than once, so independent-row confidence intervals are omitted.


In [ ]:
df['matchup_wr_decile'] = pd.qcut(df['matchup_winrate'], q=10, duplicates='drop')
calibration = (
    df.groupby('matchup_wr_decile', observed=True)
    .agg(
        player_rows=('win', 'size'),
        mean_matchup_winrate=('matchup_winrate', 'mean'),
        actual_winrate=('win', 'mean'),
    )
    .reset_index(drop=True)
)
display_percentages(calibration, ['mean_matchup_winrate', 'actual_winrate'])

x = calibration['mean_matchup_winrate'] * 100
y = calibration['actual_winrate'] * 100
lower = min(x.min(), y.min()) - 1
upper = max(x.max(), y.max()) + 1

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(x, y, 'o-', label='Observed')
ax.plot([lower, upper], [lower, upper], linestyle='--', color='gray', label='Perfect agreement')
ax.set(
    title='Actual win rate versus raw matchup win rate (descriptive)',
    xlabel='Mean Global D2+ matchup win rate (%)',
    ylabel='Actual sampled-player win rate (%)',
    xlim=(lower, upper),
    ylim=(lower, upper),
)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Results by rank

In [ ]:
def summarize_groups(frame, group_columns):
    return (
        frame.groupby(group_columns, observed=True)
        .agg(
            player_rows=('win', 'size'),
            unique_players=('puuid', 'nunique'),
            unique_matches=('match_id', 'nunique'),
            actual_winrate=('win', 'mean'),
            average_matchup_winrate=('matchup_winrate', 'mean'),
            average_otp_rate=('otp_rate', 'mean'),
            median_player_games=('player_games', 'median'),
        )
        .assign(winrate_gap=lambda table: table['actual_winrate'] - table['average_matchup_winrate'])
        .reset_index()
    )

rank_results = summarize_groups(df, ['rank'])
display_percentages(
    rank_results,
    ['actual_winrate', 'average_matchup_winrate', 'average_otp_rate', 'winrate_gap'],
    signed=['winrate_gap'],
)

rank_plot = rank_results.melt(
    id_vars='rank',
    value_vars=['actual_winrate', 'average_matchup_winrate'],
    var_name='measure',
    value_name='winrate',
)
rank_plot['measure'] = rank_plot['measure'].map({
    'actual_winrate': 'Actual',
    'average_matchup_winrate': 'Raw matchup',
})
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=rank_plot, x='rank', y='winrate', hue='measure', ax=ax)
ax.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
ax.set(title='Actual and matchup win rates by rank', xlabel='Rank', ylabel='Win rate')
plt.tight_layout()
plt.show()

## 6. Results by lane

In [ ]:
lane_results = summarize_groups(df, ['lane'])
display_percentages(
    lane_results,
    ['actual_winrate', 'average_matchup_winrate', 'average_otp_rate', 'winrate_gap'],
    signed=['winrate_gap'],
)

lane_plot = lane_results.melt(
    id_vars='lane',
    value_vars=['actual_winrate', 'average_matchup_winrate'],
    var_name='measure',
    value_name='winrate',
)
lane_plot['measure'] = lane_plot['measure'].map({
    'actual_winrate': 'Actual',
    'average_matchup_winrate': 'Raw matchup',
})
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=lane_plot, x='lane', y='winrate', hue='measure', ax=ax)
ax.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
ax.set(title='Actual and matchup win rates by lane', xlabel='Lane', ylabel='Win rate')
plt.tight_layout()
plt.show()

## 7. Results by rank and lane

The first heatmap shows actual win rate. The second subtracts the average raw matchup rate; positive values mean the sampled players won more often than their matchups alone would suggest.

In [ ]:
rank_lane_results = summarize_groups(df, ['rank', 'lane'])
display_percentages(
    rank_lane_results,
    ['actual_winrate', 'average_matchup_winrate', 'average_otp_rate', 'winrate_gap'],
    signed=['winrate_gap'],
)

actual_matrix = rank_lane_results.pivot(index='rank', columns='lane', values='actual_winrate') * 100
gap_matrix = rank_lane_results.pivot(index='rank', columns='lane', values='winrate_gap') * 100
count_matrix = rank_lane_results.pivot(index='rank', columns='lane', values='player_rows')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.heatmap(actual_matrix, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': 'Win rate (%)'}, ax=axes[0])
axes[0].set(title='Actual win rate by rank and lane', xlabel='Lane', ylabel='Rank')
sns.heatmap(gap_matrix, annot=True, fmt='+.1f', cmap='RdBu_r', center=0, cbar_kws={'label': 'Percentage-point gap'}, ax=axes[1])
axes[1].set(title='Actual minus matchup win rate', xlabel='Lane', ylabel='Rank')
plt.tight_layout()
plt.show()

print('Player-row counts behind each rank/lane cell:')
display(count_matrix.astype('Int64'))

## 8. Pre-game OTP-rate distribution

Every retained row has 50 completed earlier Ranked Solo/Duo games, so `player_games` is exactly 50. The OTP rate can change across matches for the same player as their previous-50 window moves.


In [ ]:
# Each row uses a full 50-game pre-match window; one player's OTP rate can
# change between observations as older games leave the window.
print('Pre-game window size:', df['player_games'].unique().tolist())
print('Distinct player-game OTP measurements:', len(df[['puuid', 'match_id', 'otp_rate']]))
print('Players with multiple observed matches:',
      df.groupby('puuid').size().gt(1).sum())

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=df, x='otp_rate', bins=np.linspace(0, 1, 21), ax=ax)
ax.set(title='Previous-50 champion share across player-games',
       xlabel='Champion share in previous 50 Solo/Duo games', ylabel='Player-games')
ax.xaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
plt.tight_layout()
plt.show()


## 9. Actual results across OTP-rate groups

This is descriptive rather than causal. Differences may reflect rank, lane, champion selection, and repeated observations from the same match or player.

In [ ]:
otp_bins = [0, 0.20, 0.40, 0.60, 0.80, 1.000001]
otp_labels = ['0–20%', '20–40%', '40–60%', '60–80%', '80–100%']
df['otp_rate_group'] = pd.cut(
    df['otp_rate'], bins=otp_bins, labels=otp_labels, include_lowest=True, right=False
)
otp_results = (
    df.groupby('otp_rate_group', observed=True)
    .agg(
        player_rows=('win', 'size'),
        unique_players=('puuid', 'nunique'),
        actual_winrate=('win', 'mean'),
        average_matchup_winrate=('matchup_winrate', 'mean'),
        median_player_games=('player_games', 'median'),
    )
    .assign(winrate_gap=lambda table: table['actual_winrate'] - table['average_matchup_winrate'])
    .reset_index()
)
display_percentages(
    otp_results,
    ['actual_winrate', 'average_matchup_winrate', 'winrate_gap'],
    signed=['winrate_gap'],
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(otp_results['otp_rate_group'].astype(str), otp_results['actual_winrate'] * 100, marker='o', label='Actual')
ax.plot(otp_results['otp_rate_group'].astype(str), otp_results['average_matchup_winrate'] * 100, marker='o', label='Raw matchup')
ax.set(title='Win rates across provisional OTP-rate groups', xlabel='OTP-rate group', ylabel='Win rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Side balance

In [ ]:
side_results = summarize_groups(df, ['side'])
display_percentages(
    side_results,
    ['actual_winrate', 'average_matchup_winrate', 'average_otp_rate', 'winrate_gap'],
    signed=['winrate_gap'],
)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=side_results, x='side', y='actual_winrate', ax=ax)
ax.yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
ax.set(title='Actual win rate by side', xlabel='Side', ylabel='Win rate')
plt.tight_layout()
plt.show()

## 11. Collection coverage over time

In [ ]:
df['game_date'] = df['game_start_time'].dt.strftime('%b %d')
daily_results = (
    df.groupby('game_date', sort=False)
    .agg(
        player_rows=('win', 'size'),
        unique_matches=('match_id', 'nunique'),
        actual_winrate=('win', 'mean'),
    )
    .reset_index()
)
display_percentages(daily_results, ['actual_winrate'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=daily_results, x='game_date', y='unique_matches', ax=axes[0])
axes[0].set(title='Unique matches by date', xlabel='Match date (Pacific)', ylabel='Matches')
sns.lineplot(data=daily_results, x='game_date', y='actual_winrate', marker='o', ax=axes[1])
axes[1].yaxis.set_major_formatter(lambda value, position: f'{value:.0%}')
axes[1].set(title='Actual win rate by date', xlabel='Match date (Pacific)', ylabel='Win rate')
plt.tight_layout()
plt.show()

## 12. Most represented champions

This is a coverage diagnostic—not a champion-strength ranking.

In [ ]:
top_champions = (
    df.groupby('champion')
    .agg(player_rows=('win', 'size'), unique_players=('puuid', 'nunique'), actual_winrate=('win', 'mean'))
    .sort_values('player_rows', ascending=False)
    .head(15)
    .reset_index()
)
display_percentages(top_champions, ['actual_winrate'])

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=top_champions, y='champion', x='player_rows', ax=ax)
ax.set(title='Top 15 champions by player-match observations', xlabel='Player rows', ylabel='Champion')
plt.tight_layout()
plt.show()

## Interpretation reminders

- These figures are descriptive; they do not isolate a causal effect of specialization.
- Multiple rows can come from the same match and the same player. Model uncertainty should account for clustering by both `match_id` and `puuid`.
- Every retained `otp_rate` uses 50 completed prior Solo/Duo games. Requiring a full 50-game history can select more active players; results remain observational.
- The primary model controls for raw matchup win rate, rank, lane, side, and champion when interpreting the OTP-rate coefficient.
